# Nova Story Channel — قصة + صور + فيديو + رفع (مجاني على Kaggle)
**قبل التشغيل:** Settings ← Accelerator = GPU، Internet = On، وأضف Secrets من Add-ons ← Secrets:
`GROQ_API_KEY` (إجباري) و `YT_CLIENT_ID` و `YT_CLIENT_SECRET` و `YT_REFRESH_TOKEN` (للرفع فقط).
غيّر `UPLOAD` في الخلية الأخيرة إلى True عندما تكون جاهزاً. الرفع يكون Private افتراضياً.

In [ ]:
!git clone --depth 1 -b story-channel-pipeline https://github.com/nova-api-del/nova.git /kaggle/working/nova
%cd /kaggle/working/nova
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece protobuf edge-tts requests pillow
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ('GROQ_API_KEY','YT_CLIENT_ID','YT_CLIENT_SECRET','YT_REFRESH_TOKEN'):
        try: os.environ[k] = s.get_secret(k)
        except Exception: print('secret not set:', k)
except Exception as e:
    print('Kaggle secrets unavailable:', e)

In [ ]:
# 1) اكتب القصة (Groq مجاني)
!python scripts/story_channel/write_story.py

In [ ]:
# 2) ولّد صور المشاهد بـ Z-Image Turbo (نفس الشخصية والأسلوب في كل مشهد)
import json, torch
from pathlib import Path
from diffusers import ZImagePipeline

story = json.load(open('output/story/story.json'))
char, style = story['character'], story['character']['style']
assert torch.cuda.is_available(), 'فعّل GPU من إعدادات Kaggle'
major, _ = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if major >= 8 else torch.float16
pipe = ZImagePipeline.from_pretrained('Tongyi-MAI/Z-Image-Turbo', torch_dtype=dtype, low_cpu_mem_usage=False)
pipe.enable_model_cpu_offload()

FR = Path('output/story/frames'); FR.mkdir(parents=True, exist_ok=True)
def gen(prompt, path, seed):
    img = pipe(prompt=prompt, height=576, width=1024, num_inference_steps=9, guidance_scale=0.0,
               generator=torch.Generator(device='cpu').manual_seed(seed)).images[0]
    img.save(path); del img; torch.cuda.empty_cache()

for i, sc in enumerate(story['scenes'], 1):
    who = f"Main character: {char['name']}, {char['look']}. " if sc.get('has_character', True) else ''
    gen(f"{style}. {who}Scene: {sc['visual']}", FR / f'scene_{i:02d}.png', 41000 + i)
    print(f'[{i}/{len(story["scenes"])}]', end=' ')
gen(f"{style}. Main character: {char['name']}, {char['look']}, shocked expression. Scene: {story['thumbnail_visual']}", FR / 'thumbnail.png', 49999)

In [ ]:
# 3) اصنع الفيديو (صوت إنجليزي + ترجمة + حركة كاميرا)
!python scripts/story_channel/render_story.py

In [ ]:
# 4) معاينة
from IPython.display import Video, FileLink, display
display(Video('output/story/video.mp4', embed=False, width=700)); display(FileLink('output/story/video.mp4'))

In [ ]:
# 5) رفع إلى يوتيوب (Private افتراضياً — راجعه في YouTube Studio قبل النشر)
UPLOAD = False
os.environ['YT_PRIVACY'] = 'private'
if UPLOAD:
    !python scripts/story_channel/upload_story.py
else:
    print('الرفع متوقف. غيّر UPLOAD = True عند الجاهزية.')